In [13]:

pip install pandas torch transformers datasets sacrebleu accelerate sentencepiece


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install protobuf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import sacrebleu

In [22]:
MODEL_NAME = "google/mt5-small"

MAX_LENGTH = 64          # shorter for memory
BATCH_SIZE = 4           # per-device batch size
GRAD_ACCUM = 4           # 4 * 4 = effective batch 16

HING_EPOCHS = 5
HING_LR = 1e-5

SPAN_EPOCHS = 5
SPAN_LR = 1e-5

FORCE_CPU = True


def get_device():
    if not FORCE_CPU and torch.cuda.is_available():
        return "cuda"
    if not FORCE_CPU and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = get_device()
print("Using device:", device)

Using device: cpu


In [23]:
print("Loading data...")
hing_train = pd.read_csv("data/hinglish_train.csv")
hing_val = pd.read_csv("data/hinglish_val.csv")
hing_test = pd.read_csv("data/hinglish_test.csv")

span_train = pd.read_csv("data/spanglish_train.csv")
span_val = pd.read_csv("data/spanglish_val.csv")
span_test = pd.read_csv("data/spanglish_test.csv")

print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")


Loading data...
Hinglish: 743 train, 93 val, 93 test
Spanglish: 844 train, 105 val, 106 test


In [24]:
def tokenize_df(df, tokenizer, lang_name):
    """
    Convert a (source, target) dataframe into a tokenized HF Dataset.
    We add an explicit task prefix for mT5.
    """
    sources = [
        f"translate {lang_name} to english: {text}"
        for text in df["source"].astype(str)
    ]
    targets = df["target"].astype(str).tolist()

    enc = tokenizer(
        sources,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    dec = tokenizer(
        targets,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )

    return Dataset.from_dict(
        {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": dec["input_ids"],
        }
    )

def compute_metrics(eval_pred, tokenizer):
    """
    Compute BLEU & chrF on generated sequences.
    """
    preds, labels = eval_pred

    # decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # replace -100 in labels (ignore_index) with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels]).score

    return {"bleu": bleu, "chrf": chrf}

def evaluate_model(model_path, test_df, lang_name):
    """
    Evaluate a saved mT5 model on a test dataframe.
    Returns (predictions, BLEU, chrF, ExactMatch%).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    sources = [
        f"translate {lang_name} to english: {text}"
        for text in test_df["source"].astype(str)
    ]
    refs = test_df["target"].astype(str).tolist()
    preds = []

    batch_size = 32
    for i in range(0, len(sources), batch_size):
        batch_src = sources[i:i + batch_size]
        enc = tokenizer(
            batch_src,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_length=MAX_LENGTH,
                num_beams=4,
            )

        batch_preds = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend([p.strip() for p in batch_preds])

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    em = (
        100.0
        * sum(p.lower().strip() == r.lower().strip()
              for p, r in zip(preds, refs))
        / len(refs)
    )

    return preds, bleu, chrf, em

In [25]:

print("TRAINING HINGLISH mT5 MODEL")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)
model.gradient_checkpointing_enable()

hing_train_ds = tokenize_df(hing_train, tokenizer, "hinglish")
hing_val_ds = tokenize_df(hing_val, tokenizer, "hinglish")

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

hing_args = Seq2SeqTrainingArguments(
    output_dir="models/mt5_hinglish",
    num_train_epochs=HING_EPOCHS,
    learning_rate=HING_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",          # older transformers uses eval_strategy
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False if device != "cuda" else True,
    report_to="none",
    no_cuda=(device != "cuda"),
)

hing_trainer = Seq2SeqTrainer(
    model=model,
    args=hing_args,
    train_dataset=hing_train_ds,
    eval_dataset=hing_val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, tokenizer),
)

hing_trainer.train()
hing_trainer.save_model("models/mt5_hinglish/best_model")
tokenizer.save_pretrained("models/mt5_hinglish/best_model")

print("Hinglish mT5 model saved to models/mt5_hinglish/best_model")

TRAINING HINGLISH mT5 MODEL


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/yq/pgkldf4s0ng9v2h3wf5m0chw0000gn/T/ipykernel_44033/1973362374.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  hing

Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,No log,22.426405,0.047994,2.162848
2,26.499700,21.700964,0.049096,2.166828
3,27.044000,21.039061,0.048086,2.186390
4,26.126500,20.472256,0.049327,2.202081
5,24.498600,18.736670,0.049418,2.201864


Hinglish mT5 model saved to models/mt5_hinglish/best_model


In [26]:
print("\n" + "=" * 80)
print("TRAINING SPANGLISH mT5 MODEL")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)
model.gradient_checkpointing_enable()

span_train_ds = tokenize_df(span_train, tokenizer, "spanglish")
span_val_ds = tokenize_df(span_val, tokenizer, "spanglish")

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

span_args = Seq2SeqTrainingArguments(
    output_dir="models/mt5_spanglish",
    num_train_epochs=SPAN_EPOCHS,
    learning_rate=SPAN_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False if device != "cuda" else True,
    report_to="none",
    no_cuda=(device != "cuda"),
)

span_trainer = Seq2SeqTrainer(
    model=model,
    args=span_args,
    train_dataset=span_train_ds,
    eval_dataset=span_val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, tokenizer),
)

span_trainer.train()
span_trainer.save_model("models/mt5_spanglish/best_model")
tokenizer.save_pretrained("models/mt5_spanglish/best_model")

print("Spanglish mT5 model saved to models/mt5_spanglish/best_model")



TRAINING SPANGLISH mT5 MODEL


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/yq/pgkldf4s0ng9v2h3wf5m0chw0000gn/T/ipykernel_44033/3797386880.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  span

Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,26.081300,22.278196,0.009864,1.450019
2,26.256400,21.275143,0.009929,1.451635
3,24.616700,19.730314,0.009929,1.451635
4,22.744900,16.943668,0.010083,1.474384
5,22.207500,14.949210,0.010218,1.458071


Spanglish mT5 model saved to models/mt5_spanglish/best_model


In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate(model_path, df, lang_name):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    srcs = [f"translate {lang_name} to english: {text}" for text in df['source']]
    refs = df['target'].tolist()
    preds = []

    for i in range(0, len(srcs), 32):
        batch = srcs[i:i+32]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        out = model.generate(**enc, max_length=64, num_beams=4)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    em = 100 * sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs)) / len(refs)

    return preds, bleu, chrf, em


In [30]:
# HINGLISH
marian_h_preds, m_h_bleu, m_h_chrf, m_h_em = evaluate("models/marian_hinglish/best_model", hing_test, "hinglish")
mt5_h_preds, t_h_bleu, t_h_chrf, t_h_em     = evaluate("models/mt5_hinglish/best_model", hing_test, "hinglish")

print(" Hinglish Performance:")
print(f" mT5     : BLEU={t_h_bleu:.2f}, chrF={t_h_chrf:.2f}, EM={t_h_em:.2f}%")

# SPANGLISH
marian_s_preds, m_s_bleu, m_s_chrf, m_s_em = evaluate("models/marian_spanglish/best_model", span_test, "spanglish")
mt5_s_preds, t_s_bleu, t_s_chrf, t_s_em     = evaluate("models/mt5_spanglish/best_model", span_test, "spanglish")

print("\n Spanglish Performance:")
print(f" mT5     : BLEU={t_s_bleu:.2f}, chrF={t_s_chrf:.2f}, EM={t_s_em:.2f}%")

The tokenizer you are loading from 'models/mt5_hinglish/best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


 Hinglish Performance:
 mT5     : BLEU=0.06, chrF=2.50, EM=0.00%


The tokenizer you are loading from 'models/mt5_spanglish/best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



 Spanglish Performance:
 mT5     : BLEU=0.01, chrF=2.10, EM=0.00%
